# MQTT class recommendation: pair-level and weighted embeddings

Each payload key/value pair is embedded separately. This notebook compares direct pair text, weighted key/value vector fusion, and numeric-aware fusion.

Streams are compared with one-to-one Hungarian pair matching. This prevents many unrelated fields from all matching the same generic field, which inflated the previous symmetric-best-match score.

In [ ]:
from pathlib import Path
import sys

workdir = Path.cwd()
notebook_dir = workdir if (workdir / "class_recommendation_eval.py").exists() else workdir / "notebook"
sys.path.insert(0, str(notebook_dir.resolve()))

from class_recommendation_eval import (
    DEFAULT_MODEL,
    build_pair_table,
    dataset_summary,
    load_dataset,
    load_model,
    run_pair_experiment,
)


In [ ]:
data = load_dataset()
pairs = build_pair_table(data)
print(f"Validated {len(data)} streams and {len(pairs)} key/value pairs")
display(dataset_summary(data))
display(pairs.head(10))


## Compared strategies

Direct representations:

- value only;
- key only;
- `key: value`;
- `key: value` with raw numeric values removed.

Weighted fusion:

`normalize(alpha * E(key) + (1 - alpha) * E(value))`, for alpha = 0, 0.25, 0.5, 0.75, and 1.

Numeric-aware fusion separately varies the contribution of raw numeric values while categorical values retain a 0.5 contribution.

In [ ]:
MODEL_NAME = DEFAULT_MODEL
print("Loading", MODEL_NAME)
model = load_model(MODEL_NAME)
pair_metrics, pair_cases = run_pair_experiment(data, model)
display(pair_metrics)


## Case-level analysis

The best coefficient here is only an exploratory candidate. A larger independent validation set is required before fixing it in production.

In [ ]:
best_pair = pair_metrics.iloc[0]['method']
best_pair_cases = pair_cases[pair_cases['method'] == best_pair].copy()
print("Best pair method:", best_pair)
display(best_pair_cases.sort_values(['correct', 'true_class', 'topic']))
display(
    best_pair_cases.groupby('true_class')['correct']
    .agg(['count', 'sum', 'mean'])
    .rename(columns={'sum': 'correct', 'mean': 'class_accuracy'})
)
